In [1]:
!pip install pandas requests ipywidgets jupyterlab_widgets

     ---------------------------------------- 11.4/11.4 MB 1.5 MB/s eta 0:00:00
     ---------------------------------------- 64.7/64.7 KB 1.8 MB/s eta 0:00:00
     ------------------------------------ 139.8/139.8 KB 395.0 kB/s eta 0:00:00
     -------------------------------------- 914.9/914.9 KB 1.0 MB/s eta 0:00:00
     -------------------------------------- 348.5/348.5 KB 1.1 MB/s eta 0:00:00
  Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
     -------------------------------------- 71.0/71.0 KB 980.1 kB/s eta 0:00:00
     -------------------------------------- 107.2/107.2 KB 1.2 MB/s eta 0:00:00
     -------------------------------------- 131.6/131.6 KB 1.1 MB/s eta 0:00:00
     ---------------------------------------- 2.2/2.2 MB 1.2 MB/s eta 0:00:00
  Using cached ipython-8.18.1-py3-none-any.whl (808 kB)
  Using cached traitlets-5.14.3-py3-none-any.whl (85 kB)
     ---------------------------------------- 1.2/1.2 MB 1.1 MB/s eta 0:00:00
  Using cached decorator-5.2.1-py3

You should consider upgrading via the 'C:\Users\acer\paleofauna-model\landmask\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [12]:
import os
import pygplates
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

from ipywidgets import IntSlider, VBox, Output
from IPython.display import display
import numpy as np

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

# FILE PATHS
BASE_PATH = r"C:\Users\acer\paleofauna-model\GPlates 2.5.0\GeoData\FeatureCollections\AltPlateReconstructions\Muller_etal_2022"

ROTATION_FILE = os.path.join(BASE_PATH, "1000_0_rotfile.rot")
LANDMASK_FILE = os.path.join(
    BASE_PATH,
    "COB_polygons_and_coastlines_combined_1000_0_Merdith_etal.gpml"
)

rotation_model = pygplates.RotationModel(ROTATION_FILE)
landmask_features = pygplates.FeatureCollection(LANDMASK_FILE)

LANDMASK_CACHE = "cache/landmask"
os.makedirs(LANDMASK_CACHE, exist_ok=True)

# --------------------------------------------------
# DATA LOADERS
# --------------------------------------------------

def get_plate_boundaries(time):
    if time > 410:
        paths = [
            os.path.join(BASE_PATH, '1000-410-Convergence.gpml'),
            os.path.join(BASE_PATH, '1000-410-Divergence.gpml'),
            os.path.join(BASE_PATH, '1000-410-Transforms.gpml')
        ]
    elif 250 < time <= 410:
        paths = [os.path.join(BASE_PATH, '410-250_plate_boundaries.gpml')]
    else:
        paths = [os.path.join(BASE_PATH, '250-0_plate_boundaries.gpml')]

    features = []
    for p in paths:
        features += pygplates.FeatureCollection(p)
    return features


def extract_land_polygons():
    polygons = []
    for feat in landmask_features:
        geom = feat.get_geometry()
        if geom and "Polygon" in geom.__class__.__name__:
            polygons.append(feat)
    return polygons


def reconstruct_features(features, time):
    reconstructed = []
    pygplates.reconstruct(features, rotation_model, reconstructed, time)
    return reconstructed


def reconstruct_coastlines(time):
    reconstructed = []
    pygplates.reconstruct(landmask_features, rotation_model, reconstructed, time)
    return reconstructed


def reconstruct_land_polygons(time):
    raw = extract_land_polygons()
    reconstructed = []
    pygplates.reconstruct(raw, rotation_model, reconstructed, time)
    return reconstructed

# --------------------------------------------------
# RASTERISATION FUNCTION
# --------------------------------------------------

def rasterise_landmask(
    land_features,
    rotation_model,
    time_ma,
    resolution_deg=1.0
):
    """
    Spherical landmask rasterisation using point-in-polygon tests.
    Returns:
        landmask : 2D boolean array (lat, lon)
        lats     : 1D latitude array
        lons     : 1D longitude array
    """

    cache_file = (
        f"{LANDMASK_CACHE}/"
        f"landmask_{int(time_ma)}Ma_{resolution_deg:.2f}deg.npz"
    )

    # --------------------
    # Load from cache
    # --------------------
    if os.path.exists(cache_file):
        data = np.load(cache_file)
        return data["mask"], data["lats"], data["lons"]

    # --------------------
    # Raster grid
    # --------------------
    lats = np.arange(-90, 90 + resolution_deg, resolution_deg)
    lons = np.arange(-180, 180 + resolution_deg, resolution_deg)

    lon_grid, lat_grid = np.meshgrid(lons, lats)

    # Flatten grid
    flat_lats = lat_grid.ravel()
    flat_lons = lon_grid.ravel()

    landmask_flat = np.zeros(flat_lats.shape, dtype=bool)

    # --------------------
    # Reconstruct land polygons
    # --------------------
    reconstructed = []
    pygplates.reconstruct(
        land_features,
        rotation_model,
        reconstructed,
        time_ma
    )

    polygons = []
    for feat in reconstructed:
        geom = feat.get_reconstructed_geometry()
        if isinstance(geom, pygplates.PolygonOnSphere):
            polygons.append(geom)

    # --------------------
    # Bounding-box accelerated rasterisation
    # --------------------

    for poly in polygons:

        coords = poly.to_lat_lon_list()
        if not coords:
            continue

        poly_lats, poly_lons = zip(*coords)

        min_lat = min(poly_lats)
        max_lat = max(poly_lats)
        min_lon = min(poly_lons)
        max_lon = max(poly_lons)

        # Handle dateline crossing
        crosses_dateline = (max_lon - min_lon) > 180

        # Vectorised candidate mask
        lat_mask = (flat_lats >= min_lat) & (flat_lats <= max_lat)

        if crosses_dateline:
            lon_mask = (flat_lons >= min_lon) | (flat_lons <= max_lon)
        else:
            lon_mask = (flat_lons >= min_lon) & (flat_lons <= max_lon)

        candidate_indices = np.where(lat_mask & lon_mask)[0]

        # Now only test candidates
        for idx in candidate_indices:

            if landmask_flat[idx]:
                continue

            point = pygplates.PointOnSphere(
                flat_lats[idx],
                flat_lons[idx]
            )

            if poly.is_point_in_polygon(point):
                landmask_flat[idx] = True

    # Reshape back
    landmask = landmask_flat.reshape(lat_grid.shape)

    # --------------------
    # Cache
    # --------------------
    np.savez_compressed(
        cache_file,
        mask=landmask,
        lats=lats,
        lons=lons
    )

    return landmask, lats, lons

# --------------------------------------------------
# PLOTTING HELPERS
# --------------------------------------------------

def plot_reconstructed_features(ax, reconstructed, color, linewidth=0.6):
    for feature in reconstructed:
        geom = feature.get_reconstructed_geometry()
        if geom and hasattr(geom, 'to_lat_lon_list'):
            coords = geom.to_lat_lon_list()
            if coords:
                lats, lons = zip(*coords)
                ax.plot(
                    lons, lats,
                    transform=ccrs.Geodetic(),
                    color=color,
                    linewidth=linewidth,
                    zorder=10
                )

def render_landmask(ax, time, resolution_deg=1.0):
    """
    Raster landmask renderer.
    This REPLACES all polygon filling logic.
    """

    landmask, lats, lons = rasterise_landmask(
        land_features=landmask_features,
        rotation_model=rotation_model,
        time_ma=time,
        resolution_deg=resolution_deg
    )

    from matplotlib.colors import ListedColormap

    cmap = ListedColormap([
        (0, 0, 0, 0),   # Transparent ocean
        "#89C544"
    ])

    ax.imshow(
        landmask.astype(float),
        extent=[-180, 180, -90, 90],
        origin="lower",
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=0,
        vmax=1,
        zorder=1
    )

# --------------------------------------------------
# UI
# --------------------------------------------------

out = Output()

slider = IntSlider(
    value=70,
    min=0,
    max=1000,
    step=5,
    description="Time (Ma)",
    continuous_update=False
)

def update_plot(change):
    with out:
        out.clear_output(wait=True)
        time = change["new"]

        fig = plt.figure(figsize=(12, 6))
        ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
        ax.set_global()
        ax.set_title(f"Landmask + Boundaries @ {time} Ma")

        # Landmask
        render_landmask(ax, time)

        # Plate boundaries
        boundaries = get_plate_boundaries(time)
        reconstructed_boundaries = reconstruct_features(boundaries, time)
        plot_reconstructed_features(ax, reconstructed_boundaries, color="blue")

        # Coastlines
        reconstructed_coasts = reconstruct_coastlines(time)
        plot_reconstructed_features(ax, reconstructed_coasts, color="saddlebrown", linewidth=0.4)
        
        plt.show()

slider.observe(update_plot, names="value")

display(VBox([slider, out]))
update_plot({"new": slider.value})